# ReelBench: Two-Tower + SASRec training (Kaggle/Colab GPU)

Kaggle/Colab GPU training run log for the two neural retrieval models in
[ReelBench](../README.md): the two-tower retriever and SASRec.

This replaces a corrupted notebook file that shipped in a previous commit
(invalid JSON, effectively unopenable). This notebook runs the exact same
`src.models.two_tower` / `src.models.sasrec` training and export code the
CLI scripts (`scripts/train_two_tower.py`, `scripts/train_sasrec.py`) use,
so results here match a CLI run with the same arguments.

**Before running:** upload `data/processed/train.parquet` (written by
`scripts/run_phase1.py`) to this notebook's working directory, or point
`TRAIN_PATH` below at wherever you've placed it (a mounted Google Drive
path on Colab, `/kaggle/input/...` on Kaggle).

**Runtime:** GPU (both models fall back to CPU automatically if none is
available, but training is designed for a free-tier GPU session).

## 1. Setup

In [ ]:
import os

REPO_URL = "https://github.com/<your-username>/reelbench.git"
REPO_DIR = "reelbench"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path(REPO_DIR) if Path(REPO_DIR).exists() else Path(".")
sys.path.insert(0, str(REPO_ROOT.resolve()))

TRAIN_PATH = REPO_ROOT / "data/processed/train.parquet"
OUTPUT_DIR = REPO_ROOT / "data/processed"

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints") if Path("/kaggle").exists() else Path("checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"train path: {TRAIN_PATH}")
print(f"output dir: {OUTPUT_DIR}")
print(f"checkpoint dir:{CHECKPOINT_DIR}")


In [ ]:
!pip install -q polars pyarrow mlflow
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))


In [ ]:
import polars as pl

train_df = pl.read_parquet(TRAIN_PATH)
print(f"{train_df.height:,} interactions, {train_df['userId'].n_unique():,} users, "
      f"{train_df['movieId'].n_unique():,} items")
train_df.head()


## 2. Two-tower

In-batch negative sampling, checkpointed every epoch (see
`src/models/two_tower.py`). Re-running this cell after a disconnect
resumes automatically from the last completed epoch found at
`CHECKPOINT_DIR / "two_tower.pt"`.

In [ ]:
from src.models.two_tower import export_embeddings as export_two_tower_embeddings
from src.models.two_tower import train as train_two_tower

two_tower_checkpoint = CHECKPOINT_DIR / "two_tower.pt"

two_tower_model, two_tower_id_maps = train_two_tower(
    train_df,
    checkpoint_path=two_tower_checkpoint,
    epochs=10,
    batch_size=512,
    lr=1e-3,
    embedding_dim=64,
)


In [ ]:
export_two_tower_embeddings(two_tower_model, two_tower_id_maps, OUTPUT_DIR)


## 3. SASRec

Causal self-attention over each user's chronological sequence, next-item
prediction (see `src/models/sasrec.py`). Same checkpoint-resume pattern as
above, keyed on `CHECKPOINT_DIR / "sasrec.pt"`.

In [ ]:
from src.models.sasrec import build_user_sequences
from src.models.sasrec import export_embeddings as export_sasrec_embeddings
from src.models.sasrec import train as train_sasrec

sasrec_checkpoint = CHECKPOINT_DIR / "sasrec.pt"

sasrec_model, sasrec_id_maps, sasrec_config = train_sasrec(
    train_df,
    checkpoint_path=sasrec_checkpoint,
    epochs=10,
    batch_size=128,
    lr=1e-3,
    max_seq_len=50,
    embedding_dim=64,
)


In [ ]:
# Use sasrec_config, not the literal args above: if this run resumed from a
# checkpoint trained with a different max_seq_len, train() silently used the
# checkpoint's value internally, and exporting with a mismatched value here
# would crash on a position_embedding shape mismatch. See train()'s
# docstring in src/models/sasrec.py.
sequences = build_user_sequences(train_df, sasrec_id_maps)
export_sasrec_embeddings(
    sasrec_model, sasrec_id_maps, sequences, OUTPUT_DIR,
    max_seq_len=sasrec_config["max_seq_len"],
)


## 4. Sanity-check the exported embeddings

Confirms neither export produced NaNs before you copy the parquet files
back to your local `data/processed/` (see `scripts/check_embeddings_for_nan.py`
for the same check from the command line).

In [ ]:
import numpy as np

for prefix in ["two_tower", "sasrec"]:
    for kind in ["user", "item"]:
        path = OUTPUT_DIR / f"{prefix}_{kind}_embeddings.parquet"
        df = pl.read_parquet(path)
        arr = np.array(df["embedding"].to_list())
        n_nan = int(np.isnan(arr).any(axis=1).sum())
        print(f"{path.name}: {arr.shape[0]} rows, dim {arr.shape[1]}, {n_nan} rows with NaN")


## 5. Next steps

Download `data/processed/two_tower_{user,item}_embeddings.parquet` and
`data/processed/sasrec_{user,item}_embeddings.parquet` from this session,
copy them into your local `data/processed/`, then locally run:

```bash
python scripts/build_serving_artifacts.py
python scripts/build_ui_artifacts.py
python scripts/evaluate_pipeline_models.py
```